## SEINE Video Generation model

In [1]:
!pip install munch
import zipfile
import os

zip_path = "/content/CS5260-PromptPilot-master(1).zip"  # Change this to your actual zip file path
extract_path = "/content/CS5260-PromptPilot-master"  # Change this to your desired extraction folder

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)



print("Extraction complete!")




Extraction complete!


In [2]:
!pip install omegaconf
!pip install utils
!pip install rotary_embedding_torch
!pip install xformers
!pip install pyav
!pip install transformers scipy ftfy accelerate
import torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.9 MB/s eta 0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144554 sha256=2c6e0d604914bfada7d977c232b6596adc5eb904f8a84d5aed14452077b1b9f4
  Stored in directory: /root/.cache/pip/wheels/1a/97/32/461f837398029ad76911109f07047fde1d7b661a147c7c56d1
Successfully built antlr4-python3-runtime


  Preparing metadata (setup.py) ... done
  Created wheel for utils: filename=utils-1.0.2-py2.py3-none-any.whl size=13906 sha256=a9c1fd75cd5444c7a0125517d41641efbdda9720169394ddf7cb3f9c773aa3da
  Stored in directory: /root/.cache/pip/wheels/15/0c/b3/674aea8c5d91c642c817d4d630bd58faa316724b136844094d
Successfully built utils
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!pip install --upgrade diffusers

#from diffusers import StableDiffusionPipeline
import os
import sys
import math
#sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'SEINE'))

sys.path.insert(0, '/content/CS5260-PromptPilot-master/content/CS5260-PromptPilot-master/CS5260-PromptPilot-master/SEINE')  # insert at start
from diffusion import create_diffusion

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
import argparse
import torchvision

from einops import rearrange
from models import get_models
from torchvision.utils import save_image
from diffusers.models import AutoencoderKL
from models.clip import TextEmbedder
from omegaconf import OmegaConf
from PIL import Image
import numpy as np
from torchvision import transforms
from SEINE import video_transforms
# from dataset import video_transforms

from utils import mask_generation_before
import utils

from natsort import natsorted
from diffusers.utils.import_utils import is_xformers_available
import pdb
from datetime import datetime
from huggingface_hub import snapshot_download
!pip install --upgrade diffusers huggingface_hub



model_dir = snapshot_download("CompVis/stable-diffusion-v1-4")
print("Model cached at:", model_dir)

from huggingface_hub import hf_hub_download

# This will download to the cache folder and return the local path
ckpt_path = hf_hub_download(repo_id="Vchitect/SEINE", filename="seine.pt")

print("Downloaded checkpoint to:", ckpt_path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 42.6 MB/s eta 0:00:00
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.32.2
    Uninstalling diffusers-0.32.2:
      Successfully uninstalled diffusers-0.32.2


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 33 files:   0%|          | 0/33 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Model cached at: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b


seine.pt:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Downloaded checkpoint to: /root/.cache/huggingface/hub/models--Vchitect--SEINE/snapshots/fd394e18671f986fd27eb64d827e01f6fd8e57ad/seine.pt


In [4]:
import os
import sys
import math
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'SEINE'))

import utils
from diffusion import create_diffusion

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
import argparse
import torchvision

from einops import rearrange
from models import get_models
from torchvision.utils import save_image
from diffusers.models import AutoencoderKL
from models.clip import TextEmbedder
from omegaconf import OmegaConf
from PIL import Image
import numpy as np
from torchvision import transforms
from SEINE import video_transforms
# from dataset import video_transforms
from utils import mask_generation_before
from natsort import natsorted
from diffusers.utils.import_utils import is_xformers_available
import pdb
import datetime
class SeineModel:
    def __init__(self, args):
        print('Initializing SEINE model...')

        if args.seed:
            torch.manual_seed(args.seed)
        torch.set_grad_enabled(False)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        if args.ckpt is None:
            raise ValueError("Please specify a checkpoint path using --ckpt <path>")

        # load model
        self.latent_h = args.image_size[0] // 8
        self.latent_w = args.image_size[1] // 8
        self.image_h = args.image_size[0]
        self.image_w = args.image_size[1]
        self.model = get_models(args).to(self.device)

        if args.enable_xformers_memory_efficient_attention:
            if is_xformers_available():
                self.model.enable_xformers_memory_efficient_attention()
            else:
                raise ValueError("xformers is not available. Make sure it is installed correctly")

        ckpt_path = args.ckpt
        state_dict = torch.load(ckpt_path, map_location=lambda storage, loc: storage)['ema']
        self.model.load_state_dict(state_dict)

        self.model.eval()
        pretrained_model_path = args.pretrained_model_path
        self.diffusion = create_diffusion(str(args.num_sampling_steps))
        self.vae = AutoencoderKL.from_pretrained(pretrained_model_path, subfolder="vae").to(self.device)
        self.text_encoder = TextEmbedder(pretrained_model_path).to(self.device)
        if args.use_fp16:
            # print('Warning: using half percision for inferencing!')
            self.vae.to(dtype=torch.float16)
            self.model.to(dtype=torch.float16)
            self.text_encoder.to(dtype=torch.float16)

        self.mask_type = args.mask_type
        self.num_frames = args.num_frames
        self.use_fp16 = args.use_fp16
        self.do_classifier_free_guidance = args.do_classifier_free_guidance
        self.sample_method = args.sample_method
        self.cfg_scale = args.cfg_scale
        self.use_mask = args.use_mask

        print('Initialization complete!')

    def get_input(self, input_path):
        transform_video = transforms.Compose([
                            video_transforms.ToTensorVideo(), # TCHW
                            video_transforms.ResizeVideo((self.image_h, self.image_w)),
                            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], inplace=True)
                        ])
        if input_path is not None:
            print(f'Loading video from {input_path}...')
            if os.path.isdir(input_path):
                file_list = os.listdir(input_path)
                video_frames = []
                if self.mask_type.startswith('onelast'):
                    num = int(self.mask_type.split('onelast')[-1])
                    # get first and last frame
                    first_frame_path = os.path.join(input_path, natsorted(file_list)[0])
                    last_frame_path = os.path.join(input_path, natsorted(file_list)[-1])
                    first_frame = torch.as_tensor(np.array(Image.open(first_frame_path), dtype=np.uint8, copy=True)).unsqueeze(0)
                    last_frame = torch.as_tensor(np.array(Image.open(last_frame_path), dtype=np.uint8, copy=True)).unsqueeze(0)
                    for i in range(num):
                        video_frames.append(first_frame)
                    # add zeros to frames
                    num_zeros = self.num_frames-2*num
                    for i in range(num_zeros):
                        zeros = torch.zeros_like(first_frame)
                        video_frames.append(zeros)
                    for i in range(num):
                        video_frames.append(last_frame)
                    n = 0
                    video_frames = torch.cat(video_frames, dim=0).permute(0, 3, 1, 2) # f,c,h,w
                    video_frames = transform_video(video_frames)
                else:
                    for file in file_list:
                        if file.endswith('jpg') or file.endswith('png'):
                            image = torch.as_tensor(np.array(Image.open(file), dtype=np.uint8, copy=True)).unsqueeze(0)
                            video_frames.append(image)
                        else:
                            continue
                    n = 0
                    video_frames = torch.cat(video_frames, dim=0).permute(0, 3, 1, 2) # f,c,h,w
                    video_frames = transform_video(video_frames)
                return video_frames, n
            elif os.path.isfile(input_path):
                _, full_file_name = os.path.split(input_path)
                file_name, extension = os.path.splitext(full_file_name)
                if extension == '.jpg' or extension == '.png':
                    print("Loading the input image...")
                    video_frames = []
                    num = int(self.mask_type.split('first')[-1])
                    first_frame = torch.as_tensor(np.array(Image.open(input_path).convert('RGB'), dtype=np.uint8, copy=True)).unsqueeze(0)
                    for i in range(num):
                        video_frames.append(first_frame)
                    num_zeros = self.num_frames-num
                    for i in range(num_zeros):
                        zeros = torch.zeros_like(first_frame)
                        video_frames.append(zeros)
                    n = 0
                    video_frames = torch.cat(video_frames, dim=0).permute(0, 3, 1, 2) # f,c,h,w
                    video_frames = transform_video(video_frames)
                    return video_frames, n
                else:
                    raise TypeError(f'{extension} is not supported !!')
            else:
                raise ValueError('Please check your path input!!')
        else:
            raise ValueError('Need to give a video or some images')

    def auto_inpainting(self, video_input, masked_video, mask, prompt, negative_prompt):
        b,f,c,h,w = video_input.shape

        # prepare inputs
        if self.use_fp16:
            z = torch.randn(1, 4, self.num_frames, self.latent_h, self.latent_w, dtype=torch.float16, device=self.device) # b,c,f,h,w
            masked_video = masked_video.to(dtype=torch.float16)
            mask = mask.to(dtype=torch.float16)
        else:
            z = torch.randn(1, 4, self.num_frames, self.latent_h, self.latent_w, device=self.device) # b,c,f,h,w

        masked_video = rearrange(masked_video, 'b f c h w -> (b f) c h w').contiguous()
        masked_video = self.vae.encode(masked_video).latent_dist.sample().mul_(0.18215)
        masked_video = rearrange(masked_video, '(b f) c h w -> b c f h w', b=b).contiguous()
        mask = torch.nn.functional.interpolate(mask[:,:,0,:], size=(self.latent_h, self.latent_w)).unsqueeze(1)

        # classifier_free_guidance
        if self.do_classifier_free_guidance:
            masked_video = torch.cat([masked_video] * 2)
            mask = torch.cat([mask] * 2)
            z = torch.cat([z] * 2)
            prompt_all = [prompt] + [negative_prompt]

        else:
            masked_video = masked_video
            mask = mask
            z = z
            prompt_all = [prompt]

        text_prompt = self.text_encoder(text_prompts=prompt_all, train=False)
        model_kwargs = dict(encoder_hidden_states=text_prompt,
                                class_labels=None,
                                cfg_scale=self.cfg_scale,
                                use_fp16=self.use_fp16,) # tav unet

        # sample video
        if self.sample_method == 'ddim':
            samples = self.diffusion.ddim_sample_loop(
                self.model.forward_with_cfg, z.shape, z, clip_denoised=False, model_kwargs=model_kwargs, progress=True, device=self.device, \
                mask=mask, x_start=masked_video, use_concat=self.use_mask
            )
        elif self.sample_method == 'ddpm':
            samples = self.diffusion.p_sample_loop(
                self.model.forward_with_cfg, z.shape, z, clip_denoised=False, model_kwargs=model_kwargs, progress=True, device=self.device, \
                mask=mask, x_start=masked_video, use_concat=self.use_mask
            )
        samples, _ = samples.chunk(2, dim=0) # [1, 4, 16, 32, 32]
        if self.use_fp16:
            samples = samples.to(dtype=torch.float16)

        video_clip = samples[0].permute(1, 0, 2, 3).contiguous() # [16, 4, 32, 32]
        video_clip = self.vae.decode(video_clip / 0.18215).sample # [16, 3, 256, 256]

        return video_clip

    def generate_video(self, args):
        prompt = args.text_prompt

        if prompt == []:
            prompt = args.input_path.split('/')[-1].split('.')[0].replace('_', ' ')
        else:
            prompt = prompt[0]
        prompt_base = prompt.replace(' ','_')

        if not os.path.exists(os.path.join(args.save_path)):
            os.makedirs(os.path.join(args.save_path))
        video_input, reserve_frames = self.get_input(args.input_path) # f,c,h,w
        video_input = video_input.to(self.device).unsqueeze(0)  # b,f,c,h,w

        mask = mask_generation_before(self.mask_type, video_input.shape, video_input.dtype, self.device) # b,f,c,h,w
        masked_video = video_input * (mask == 0)

        video_clip = self.auto_inpainting(video_input, masked_video, mask, prompt, args.negative_prompt)
        video_ = ((video_clip * 0.5 + 0.5) * 255).add_(0.5).clamp_(0, 255).to(dtype=torch.uint8).cpu().permute(0, 2, 3, 1)
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{prompt_base}_{timestamp}.mp4"
        save_video_path = os.path.join(args.save_path, filename)
        torchvision.io.write_video(save_video_path, video_, fps=8, video_codec="mpeg4")
        print(f'Video saved in {save_video_path}')
        return save_video_path


## Load SEINE model with configs

In [5]:
parser = argparse.ArgumentParser()
parser.add_argument("--config", type=str, default="/content/CS5260-PromptPilot-master/content/CS5260-PromptPilot-master/CS5260-PromptPilot-master/configs/seine.yaml")
args, unknown = parser.parse_known_args()
omega_conf = OmegaConf.load(args.config)
seine_model = SeineModel(omega_conf)

Initializing SEINE model...
Initialization complete!


## PromptPilot Agent

In [26]:
from openai import OpenAI
import yaml
import re
from argparse import Namespace
import base64
import torch
from PIL import Image
from typing import List, Dict
from transformers import (
    CLIPProcessor, CLIPModel,
    Blip2Processor, Blip2ForConditionalGeneration
)
import torchvision.transforms as T
import cv2
import numpy as np
from PIL import Image

class PromptPilotAgent:
    def __init__(self, seine_model,device="cuda"):
        self.llm = OpenAI(
                api_key="fy3jHNMV7OC7t7fQprQkFgp7NeSlRsMG",
                base_url="https://api.deepinfra.com/v1/openai",
            )
        self.temperature = 0.0
        self.max_iterations = 10
        self.seine_model = seine_model


        self.SYSTEM_PROMPT = """
                                You are a prompt refinement agent specialized in enhancing prompts for video generation models that take a single image as input.
                                Your goal is to improve the given prompt so that the generated video is visually coherent, smooth in motion, and logically consistent with the content and context of the image.
                                Carefully consider the visual elements and implied actions in the image, and rewrite the prompt to guide the model toward generating a realistic and temporally logical video sequence.
                             """

        self.USER_PROMPT = """
                              "Given an image and the previous prompt, refine the prompt to result in a better video."
                           """
        # Given an image and the previous prompt, refine the prompt and also create a negative prompt to result in a better video. Output the refined prompt and negative prompt in the following format:
        #                       Refined prompt: <refined prompt>
        #                       Negative prompt: <negative prompt>

        self.messages = [{"role": "system", "content": self.SYSTEM_PROMPT}]

        ##ADD
        self.device = device
        self.repeat=1

        # Load CLIP
        self.clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
        self.clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        #openai/clip-vit-large-patch14

        # Load BLIP-2

        #self.blip_model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b").to(device)
        #self.blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
        #Salesforce/blip2-flan-t5-xl


        self.resize = T.Resize((224, 224))
        self.to_tensor = T.ToTensor()
        self.scores = {"clip_tva_score": 0,
            #"blip2_summary": 0,
            #"blip2_tva_score": 0,
            "temporal_consistency": 0,
            "dynamic_degree": 0}
        self.threshold = [0.6, 0.3, 0.3]

        ##END
    def extract_frames(self,video_path: str, num_frames: int = 4) -> List[Image.Image]:
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_idxs = np.linspace(0, total_frames - 1, num_frames).astype(int)

        frames = []
        for idx in frame_idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(Image.fromarray(rgb_frame))
        cap.release()
        return frames

    def compute_clip_alignment(self, frames: List[Image.Image], prompt: str) -> float:
        inputs = self.clip_processor(
            text=[prompt] * len(frames), images=frames,
            return_tensors="pt", padding=True
        ).to(self.device)
        with torch.no_grad():
            outputs = self.clip_model(**inputs)
            sims = torch.cosine_similarity(outputs.image_embeds, outputs.text_embeds)
        return sims.mean().item()

    #def generate_blip_summary(self, frame: Image.Image) -> str:
        #inputs = self.blip_processor(
            #images=frame, text=" Describe this scene.",
            #return_tensors="pt"
        #).to(self.device)
        #print(inputs)

        ##
        #summaries = []
        #for frame in frames:
            #inputs = self.blip_processor(images=frame, text=None,return_tensors="pt").to(self.device)
            #with torch.no_grad():
                  #output = self.blip_model.generate(**inputs)
            #caption = self.blip_processor.decode(output[0], skip_special_tokens=True)
            #summaries.append(caption)
        #print("summaries",summaries)
        #return summaries
        ##


        ##with torch.no_grad():
            ##output = self.blip_model.generate(**inputs)
            ##print("output",self.blip_processor.decode(output[0], skip_special_tokens=True))
        ##return self.blip_processor.decode(output[0], skip_special_tokens=True)

    def compare_blip_to_prompt(self, summary: str, prompt: str) -> float:
        clip_inputs = self.clip_processor(
            text=[prompt, summary], images=None, return_tensors="pt", padding=True
        ).to(self.device)
        with torch.no_grad():
            txt_embs = self.clip_model.get_text_features(**clip_inputs)
            sim = torch.cosine_similarity(txt_embs[0].unsqueeze(0), txt_embs[1].unsqueeze(0))
        return sim.item()

    def compute_temporal_consistency(self, frames: List[Image.Image]) -> float:
        gray_frames = [cv2.cvtColor(np.array(f), cv2.COLOR_RGB2GRAY) for f in frames]
        total_flow = 0.0
        for i in range(1, len(gray_frames)):
            flow = cv2.calcOpticalFlowFarneback(
                gray_frames[i - 1], gray_frames[i], None,
                pyr_scale=0.5, levels=3, winsize=15, iterations=3,
                poly_n=5, poly_sigma=1.2, flags=0
            )
            magnitude = np.linalg.norm(flow, axis=2).mean()
            total_flow += magnitude
        return total_flow / (len(frames) - 1)

    def compute_dynamic_degree(self, frames: List[Image.Image]) -> float:
        return self.compute_temporal_consistency(frames)  # Simple proxy for DD

    def evaluate_video(self, frames: List[Image.Image], prompt: str) -> Dict[str, float]:
        print("frames",frames)
        ##middle_frame = frames[len(frames) // 2]
        clip_score = self.compute_clip_alignment(frames, prompt)
        #blip_summary = self.generate_blip_summary(frames)
        #blip_score = self.compare_blip_to_prompt(blip_summary, prompt)
        tc_score = self.compute_temporal_consistency(frames)
        dd_score = self.compute_dynamic_degree(frames)

        self.scores.update({
        "clip_tva_score": clip_score,
        "temporal_consistency": tc_score,
        "dynamic_degree": dd_score
          })

        return self.scores  # Return updated dictionary if needed
    def set_first_text_prompt(self, yaml_path):
        with open(yaml_path, 'r') as f:
            content = yaml.safe_load(f)

        # Extract the input_path value to use as the new_prompt
        input_path = content.get("input_path", "No input_path found")
        prompt = input_path.split('/')[-1].split('.')[0].replace('_', ' ')

        with open(yaml_path, 'r') as f:
            content = f.read()

        # replace the text_prompt line
        new_content = re.sub(
            r'text_prompt:.*?\n',
            f'text_prompt: [{prompt}]\n',
            content
        )

        with open(yaml_path, 'w') as f:
            print("First",new_content)
            f.write(new_content)

    def update_text_prompt(self, yaml_path, refined_prompt): #, negative_prompt):
        with open(yaml_path, 'r') as f:
            content = f.read()

        # replace the text_prompt line
        print("REFINED_PROMPT",refined_prompt)
        new_content = re.sub(
            r'text_prompt:.*?\n',
            f'text_prompt: [{refined_prompt}]\n',
            content
        )

        # NOTE: not using this because output is worse
        # replace the negative_prompt line
        # new_content = re.sub(
        #     r'negative_prompt:\s*""',
        #     f'negative_prompt: "{negative_prompt}"',
        #     new_content
        # )

        with open(yaml_path, 'w') as f:
            f.write(new_content)

    def query_llm(self):
        response = self.llm.chat.completions.create(
            # model="meta-llama/Llama-3.2-11B-Vision-Instruct",
            model="meta-llama/Llama-3.2-90B-Vision-Instruct",
            messages=self.messages,
            temperature=self.temperature
        )
        return response.choices[0].message.content

    def clean_response(self, response):
    #     refined_match = re.search(r"Refined prompt:\s*((?:.|\n)*?)(?=\n\s*Negative prompt:|\Z)", response)
    #     negative_match = re.search(r"Negative prompt:\s*((?:.|\n)*?)\Z", response)
        refined_match = re.search(
        r"Here's a refined prompt:\s*\"([^\"]+)\"",
        response
    )

        refined_prompt = refined_match.group(1).strip().replace('\n', ' ') if refined_match else ""
        # negative_prompt = negative_match.group(1).strip().replace('\n', ' ') if negative_match else ""

        print("Refined:", refined_prompt)
        # print("Negative:", negative_prompt)

        return refined_prompt #, negative_prompt

    def cleanup(self):
        self.messages = [{"role": "system", "content": self.SYSTEM_PROMPT}]
        torch.cuda.empty_cache()

    def run(self, yaml_path):
        print("Generating video...")

        # set text prompt for the first time
        self.set_first_text_prompt(yaml_path)

        # Read the file content
        with open(yaml_path, 'r') as f:
            content = yaml.safe_load(f)
        ##current_prompt = content.get("text_prompt", [""])[0] ##added
        ##ADD
        with open(yaml_path, "r") as f:
              content = yaml.safe_load(f)
        args = Namespace(**content)
        ##END
        for iteration in range(self.max_iterations):
            with open(yaml_path, 'r') as f:
                content = yaml.safe_load(f)
            current_prompt = content.get("text_prompt", [""])[0] ##added
            print(f"\nIteration {iteration+1}/{self.max_iterations}")
            print("Current prompt:", current_prompt)
            video_path = self.seine_model.generate_video(args)
            frames = self.extract_frames(video_path)
            scores = self.evaluate_video(frames, current_prompt)
            print("Scores:", scores)
            if all(score >= threshold for score, threshold in zip(scores.values(), self.threshold)):
                print("All scores meet threshold. Stopping refinement.")
                break
            input_path = content.get("input_path", None)
            with open(input_path, "rb") as image_file:
                image_base64 = base64.b64encode(image_file.read()).decode("utf-8")
            text = self.USER_PROMPT + \
                f"""
                    Previous prompt: {current_prompt}
                    Refined prompt:
                """
                    # Negative prompt:

            # create llm use prompt
            self.messages = [
                {
                    "role": "user",
                    "content": text  # all text content here
                }
            ]

            print("🔍 Type of self.messages:", type(self.messages))
            print("🔍 Value of self.messages:", self.messages)
            llm_response = self.query_llm()
            print("llm_response",llm_response)
            refined_prompt = self.clean_response(llm_response)
            print("refined_prompt",refined_prompt)
            self.update_text_prompt(yaml_path, refined_prompt)
            current_prompt = refined_prompt


In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Create Prompt Agent

In [28]:
agent = PromptPilotAgent(seine_model=seine_model)

directory = "/content/CS5260-PromptPilot-master/content/CS5260-PromptPilot-master/CS5260-PromptPilot-master/configs/images"

# Using os.walk to loop through subdirectories
yaml_paths = []
for subdir, _, files in os.walk(directory):
    for file in files:
        if file.endswith('.yaml'):
            # Add the full path of each YAML file
            yaml_paths.append(os.path.join(subdir, file))
            print(os.path.join(subdir, file))

# Process each YAML file
for yaml_path in yaml_paths:
    print(f"\nProcessing: {yaml_path}")
    agent.run(yaml_path)

/content/CS5260-PromptPilot-master/content/CS5260-PromptPilot-master/CS5260-PromptPilot-master/configs/images/sea.yaml
/content/CS5260-PromptPilot-master/content/CS5260-PromptPilot-master/CS5260-PromptPilot-master/configs/images/pouring-essense.yaml

Processing: /content/CS5260-PromptPilot-master/content/CS5260-PromptPilot-master/CS5260-PromptPilot-master/configs/images/sea.yaml
Generating video...
First input_path: "/content/CS5260-PromptPilot-master/content/CS5260-PromptPilot-master/CS5260-PromptPilot-master/dataset/beauty_of_the_sea.png"
save_path: "./results/sea"

text_prompt: [beauty of the sea]
negative_prompt: ""


Iteration 1/10
Current prompt: beauty of the sea
Loading video from /content/CS5260-PromptPilot-master/content/CS5260-PromptPilot-master/CS5260-PromptPilot-master/dataset/beauty_of_the_sea.png...
Loading the input image...


  0%|          | 0/250 [00:00<?, ?it/s]

Video saved in ./results/sea/beauty_of_the_sea_20250424_084401.mp4
frames [<PIL.Image.Image image mode=RGB size=560x240 at 0x7B33C0173790>, <PIL.Image.Image image mode=RGB size=560x240 at 0x7B33B8956090>, <PIL.Image.Image image mode=RGB size=560x240 at 0x7B33B8BA7990>, <PIL.Image.Image image mode=RGB size=560x240 at 0x7B33AA3CFB50>]
Scores: {'clip_tva_score': 0.29696932435035706, 'temporal_consistency': np.float32(4.2827077), 'dynamic_degree': np.float32(4.2827077)}
🔍 Type of self.messages: <class 'list'>
🔍 Value of self.messages: [{'role': 'user', 'content': '\n                              "Given an image and the previous prompt, refine the prompt to result in a better video."\n                           \n                    Previous prompt: beauty of the sea\n                    Refined prompt:\n                '}]
llm_response Here's a refined prompt:

"Serenely capture the majestic beauty of the ocean's waves, vibrant coral reefs, and diverse marine life, with a warm golden light

IndexError: list index out of range

In [ ]:
#!zip -r CS5260-PromptPilot-master.zip /content/CS5260-PromptPilot-master

In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


video_path = "/content/results/essence_is_poured_from_a_bottle/A_bottle_of_oil_is_carefully_tilted_20250420_074516.mp4"
frames = extract_frames(video_path,2)
prompt = "A wave crashes into the ocean, its crest breaking and foaming as it rolls towards the shore, the sunlight dancing across the water's surface and creating a shimmering effect."

aligner = VideoVLMAligner(device="cuda" if torch.cuda.is_available() else "cpu")
scores = aligner.evaluate_video(frames, prompt)

# Print results
for k, v in scores.items():
    print(f"{k}: {v}")

In [ ]:

          results = {key: value > self.threshold[idx] for idx, (key, value) in enumerate(self.scores.items())}
          if False in results.values():
              repeat = 1
          else:
              repeat = 0
              break

          # get the string inside the text_prompt list
          prompt = content.get("text_prompt", [None])[0]
          print("First prompt:",prompt)
          input_path = content.get("input_path", None)

          # prepare llm image prompt
          with open(input_path, "rb") as image_file:
              encoded_image = base64.b64encode(image_file.read()).decode("utf-8")

          # prepare llm text prompt
          text = self.USER_PROMPT + \
                f"""
                    Previous prompt: {prompt}
                    Refined prompt:
                """
                    # Negative prompt:

          # create llm use prompt
          self.messages.append(
            {
                "role": "user",
                "content": [
                      {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64, {encoded_image}"
                          }
                      },
                      {
                        "type": "text",
                        "text": text
                      }
                  ]
              }
          )

          # get llm response
          response = self.query_llm()
          print("response:",response)

          # clean up response
          # refined_prompt, negative_prompt = self.clean_response(response)
          refined_prompt = self.clean_response(response)

          # update yaml
          self.update_text_prompt(yaml_path, refined_prompt) #, negative_prompt)

          # prepare args for video generation
          with open(yaml_path, "r") as f:
              content = yaml.safe_load(f)
          args = Namespace(**content)
          Video_path = self.seine_model.generate_video(args)

          # cleanup after generation
          self.cleanup()

          frames = self.extract_frames(Video_path,2)
          prompt = response

          scores = self.evaluate_video(frames,prompt)
          print(scores)
          for k, v in scores.items():
              print(f"{k}: {v}")
          # compute evaluation metrics
          # 3 scores...

          # feed this as input to LLM for next refinement

          # cleanup after generation
          self.cleanup()